In [9]:
import pandas as pd

df = pd.read_csv("mails_dataset.csv")

print("Distribución Sentiment:")
print(df['sentiment'].value_counts())

print("\nDistribución Priority:")
print(df['priority'].value_counts())

print("\nDistribución Category:")
print(df['category'].value_counts())


Distribución Sentiment:
sentiment
neutro      100
positivo     82
negativo     69
Name: count, dtype: int64

Distribución Priority:
priority
media    100
baja      78
alta      73
Name: count, dtype: int64

Distribución Category:
category
solicitud    78
queja        64
otro         58
comercial    51
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np
import torch
from transformers import EarlyStoppingCallback

df = pd.read_csv("mails_dataset.csv")  # Ajusta la ruta si hace falta

df['text_combined'] = df['subject'].fillna('') + ' </s> ' + df['text'].fillna('')

# Nos centramos en la columna combinada y la etiqueta priority
df = df[['text_combined', 'priority']].dropna()

# Codificar etiquetas
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["priority"])

# Guardar las etiquetas para decodificar luego
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}
id2label = {i: label for label, i in label2id.items()}

#Crear Dataset de Hugging Face
dataset = Dataset.from_pandas(df.rename(columns={"text_combined": "text", "label": "label"}))

#Tokenización
model_name = "pysentimiento/robertuito-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

dataset = dataset.map(tokenize, batched=True)

#División entrenamiento y validación
dataset = dataset.train_test_split(test_size=0.2)

#Cargar modelo preentrenado
num_labels = len(label2id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

#Configurar entrenamiento
training_args = TrainingArguments(
    output_dir="./priority_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,      
    metric_for_best_model="f1",        
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    logging_dir="./logs_priority",
    logging_steps=10
)

#Función de evaluación
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

#Entrenador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

#Entrenar
trainer.train()

#Guardar el modelo y tokenizer
trainer.save_model("./priority_model")
tokenizer.save_pretrained("./priority_model")


c:\Users\Aaron\OneDrive\Escritorio\Proyecto\MailClassifier\.venv\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at pysentimiento/robertuito-base-uncased and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/150 [00:00<?, ?it/s]

c:\Users\Aaron\OneDrive\Escritorio\Proyecto\MailClassifier\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

c:\Users\Aaron\OneDrive\Escritorio\Proyecto\MailClassifier\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.3922, 'grad_norm': 8.469736099243164, 'learning_rate': 2.666666666666667e-06, 'epoch': 5.2}
{'loss': 0.3441, 'grad_norm': 5.596838474273682, 'learning_rate': 1.3333333333333334e-06, 'epoch': 5.6}
{'loss': 0.2637, 'grad_norm': 6.995255947113037, 'learning_rate': 0.0, 'epoch': 6.0}


  0%|          | 0/7 [00:00<?, ?it/s]

{'eval_loss': 0.4504396915435791, 'eval_accuracy': 0.803921568627451, 'eval_f1': 0.7998767203358308, 'eval_runtime': 2.6619, 'eval_samples_per_second': 19.159, 'eval_steps_per_second': 2.63, 'epoch': 6.0}
{'train_runtime': 333.3675, 'train_samples_per_second': 3.6, 'train_steps_per_second': 0.45, 'train_loss': 0.5651565027236939, 'epoch': 6.0}


('./priority_model\\tokenizer_config.json',
 './priority_model\\special_tokens_map.json',
 './priority_model\\tokenizer.json')

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(dataset["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print("Reporte de clasificación por clase:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))


c:\Users\Aaron\OneDrive\Escritorio\Proyecto\MailClassifier\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/7 [00:00<?, ?it/s]

Reporte de clasificación por clase:
              precision    recall  f1-score   support

        alta       0.92      1.00      0.96        12
        baja       0.76      0.84      0.80        19
       media       0.82      0.70      0.76        20

    accuracy                           0.82        51
   macro avg       0.84      0.85      0.84        51
weighted avg       0.82      0.82      0.82        51

